# Chapter 4 | Student Profiling and Personalised Learning through Unsupervised Learning

**Companion notebook** · Behavioural feature analysis · K-Means clustering · Educational interpretation

This notebook follows the analytical sequence in Chapter 4 of the accompanying book. It covers exploratory analysis, standardisation, cluster selection, six-cluster modelling, visual validation, post-hoc performance profiles and educational interpretation. Figures **4.2–4.12** are generated from the dataset and saved as both PDF and PNG. Figure 4.1 is the conceptual framework supplied separately in the book.

**Dataset:** Adil Shamim, *Student Performance and Learning Behavior Dataset*, [Kaggle dataset page](https://www.kaggle.com/datasets/adilshamim8/student-performance-and-learning-style/data). Download the original `student_performance.csv` and place it beside this notebook or inside a `data/` directory. The notebook also detects the file when mounted as a Kaggle notebook input; an optional `kagglehub` download route is supported when the package and access are available.

**Interpretation:** The dataset comprises coded behavioural, contextual, and outcome variables. Its original sampling and collection procedures are not established by this analysis. Clusters describe patterns in the recorded data, not fixed student characteristics. Recorded learning-style categories are descriptive and do not justify assigning students to fixed teaching modalities. The analysis does not establish the effectiveness of any instructional intervention.

**Required packages:** `numpy`, `pandas`, `matplotlib`, `seaborn`, and `scikit-learn`. The optional download route uses `kagglehub`. Run the notebook from the top after making the CSV available.

## 1. Dataset and exploratory analysis

<div style="background-color:#F3F7FA;border-left:4px solid #476B82;border-radius:5px;padding:11px 14px;margin:15px 0 7px;">
<strong style="color:#253F50;">Libraries and output settings</strong><div style="font-size:0.94em;color:#485A65;margin-top:3px;">Load the statistical and plotting tools, set reproducible model parameters, and define one figure-export function.</div>
</div>

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
K_SELECTED = 6
K_CANDIDATES = list(range(2, 11))
SILHOUETTE_SAMPLE_SIZE = 2500
OUTPUT_DIR = Path('figures')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams.update({
    'figure.dpi': 110, 'savefig.dpi': 300,
    'axes.titlesize': 13, 'axes.labelsize': 11,
    'xtick.labelsize': 10, 'ytick.labelsize': 10,
    'pdf.fonttype': 42, 'ps.fonttype': 42,
})


def save_figure(fig, figure_id, short_name):
    """Export one book-aligned figure as scalable PDF and high-resolution PNG."""
    stem = f'figure_{figure_id}_{short_name}'
    for ext in ('pdf', 'png'):
        fig.savefig(OUTPUT_DIR / f'{stem}.{ext}', bbox_inches='tight', facecolor='white')
    plt.show()
    plt.close(fig)
    print(f'Saved: {stem}.pdf and {stem}.png')

<div style="background-color:#F3F7FA;border-left:4px solid #476B82;border-radius:5px;padding:11px 14px;margin:15px 0 7px;">
<strong style="color:#253F50;">Load the original dataset</strong><div style="font-size:0.94em;color:#485A65;margin-top:3px;">Read the Kaggle CSV without changing its records; show the observation count and the supplied columns.</div>
</div>

In [ ]:
DATASET_PAGE = (
    'https://www.kaggle.com/datasets/'
    'adilshamim8/student-performance-and-learning-style/data'
)

local_candidates = [
    Path('student_performance.csv'),
    Path('data/student_performance.csv'),
    Path('datasets/student_performance.csv'),
]
local_candidates += sorted(Path('/kaggle/input').glob('**/student_performance.csv')) if Path('/kaggle/input').exists() else []

csv_path = next((p for p in local_candidates if p.is_file()), None)
if csv_path is None:
    try:
        import kagglehub
        downloaded = Path(kagglehub.dataset_download(
            'adilshamim8/student-performance-and-learning-style'
        ))
        matches = sorted(downloaded.rglob('student_performance.csv'))
        csv_path = matches[0] if matches else None
    except (ImportError, OSError, RuntimeError, ValueError) as exc:
        print(f'Optional Kaggle download unavailable: {exc}')

if csv_path is None:
    raise FileNotFoundError(
        'Download student_performance.csv from ' + DATASET_PAGE +
        ' and place it next to this notebook or in the data/ folder.'
    )

student_df = pd.read_csv(csv_path)
student_df.columns = student_df.columns.str.strip()
print('Source file:', csv_path)
print('Records and variables:', student_df.shape)
display(student_df.head())

<div style="background-color:#F3F7FA;border-left:4px solid #476B82;border-radius:5px;padding:11px 14px;margin:15px 0 7px;">
<strong style="color:#253F50;">Check data structure and completeness</strong><div style="font-size:0.94em;color:#485A65;margin-top:3px;">Verify all expected variables and report missing or repeated records without silently altering the dataset.</div>
</div>

In [ ]:
EXPECTED_COLUMNS = [
    'StudyHours', 'Attendance', 'Resources', 'Extracurricular',
    'Motivation', 'Internet', 'Gender', 'Age', 'LearningStyle',
    'OnlineCourses', 'Discussions', 'AssignmentCompletion',
    'ExamScore', 'EduTech', 'StressLevel', 'FinalGrade',
]

missing_columns = sorted(set(EXPECTED_COLUMNS) - set(student_df.columns))
if missing_columns:
    raise ValueError('Missing required columns: ' + ', '.join(missing_columns))

original_count = len(student_df)
working_df = student_df[EXPECTED_COLUMNS].copy()
for variable in EXPECTED_COLUMNS:
    working_df[variable] = pd.to_numeric(working_df[variable], errors='coerce')

missing_values = working_df.isna().sum()
duplicate_rows = int(working_df.duplicated().sum())
print('Missing values by variable:')
display(missing_values.to_frame('Missing count').T)
print('Repeated full records:', duplicate_rows)
print('Recorded observations:', original_count)
if original_count != 14003:
    print('Note: this file differs in size from the 14,003-record version described in the book.')

if missing_values.any():
    raise ValueError(
        'The supplied file contains missing or non-numeric values. '
        'Review the data before clustering; no values are silently imputed.'
    )

# Preserve repeated rows, matching the supplied source workflow.

<div style="background-color:#F3F7FA;border-left:4px solid #476B82;border-radius:5px;padding:11px 14px;margin:15px 0 7px;">
<strong style="color:#253F50;">Figure 4.2 | Cohort composition</strong><div style="font-size:0.94em;color:#485A65;margin-top:3px;">Examine the frequency of the recorded gender codes without assuming that a code maps to a particular label.</div>
</div>

In [ ]:
fig, ax = plt.subplots(figsize=(7.3, 4.6))
sns.countplot(data=working_df, x='Gender', color='#436F83', ax=ax)
ax.set(title='Gender Distribution in the Dataset', xlabel='Gender (coded categories)', ylabel='Number of Records')
fig.tight_layout()
save_figure(fig, '4_2', 'gender_distribution')

<div style="background-color:#F3F7FA;border-left:4px solid #476B82;border-radius:5px;padding:11px 14px;margin:15px 0 7px;">
<strong style="color:#253F50;">Figure 4.3 | Study-time variation</strong><div style="font-size:0.94em;color:#485A65;margin-top:3px;">Describe variation in reported weekly study hours before distance-based segmentation.</div>
</div>

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.7))
sns.histplot(data=working_df, x='StudyHours', bins=20, kde=True, color='#436F83', ax=ax)
ax.set(title='Distribution of Weekly Study Hours', xlabel='Study Hours per Week', ylabel='Number of Records')
fig.tight_layout()
save_figure(fig, '4_3', 'study_hours_distribution')

<div style="background-color:#F3F7FA;border-left:4px solid #476B82;border-radius:5px;padding:11px 14px;margin:15px 0 7px;">
<strong style="color:#253F50;">Figure 4.4 | Learning-style indicators</strong><div style="font-size:0.94em;color:#485A65;margin-top:3px;">Compare outcome distributions across recorded preference codes; the chart does not establish fixed learning types.</div>
</div>

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.7))
sns.boxplot(data=working_df, x='LearningStyle', y='ExamScore', color='#97BEC9', ax=ax)
ax.set(title='Exam Score Distribution by Learning Style',
       xlabel='Learning Style (coded categories)', ylabel='Exam Score')
fig.tight_layout()
save_figure(fig, '4_4', 'learning_style_exam_score')

<div style="background-color:#F3F7FA;border-left:4px solid #476B82;border-radius:5px;padding:11px 14px;margin:15px 0 7px;">
<strong style="color:#253F50;">Figure 4.5 | Variable associations</strong><div style="font-size:0.94em;color:#485A65;margin-top:3px;">Inspect pairwise numerical associations and potential redundancy; correlations involving category codes require caution.</div>
</div>

In [ ]:
correlations = working_df.corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(11.5, 9))
sns.heatmap(correlations, cmap='vlag', center=0, vmin=-1, vmax=1,
            linewidths=0.3, cbar_kws={'label': 'Pearson correlation'}, ax=ax)
ax.set_title('Correlation Matrix of Numeric Features')
ax.tick_params(axis='x', labelrotation=65)
ax.tick_params(axis='y', labelrotation=0)
fig.tight_layout()
save_figure(fig, '4_5', 'numeric_correlation_matrix')

## 2. Behavioural features and cluster selection

<div style="background-color:#F3F7FA;border-left:4px solid #476B82;border-radius:5px;padding:11px 14px;margin:15px 0 7px;">
<strong style="color:#253F50;">Feature selection and standardisation</strong><div style="font-size:0.94em;color:#485A65;margin-top:3px;">Use the nine features reported in the chapter. Exclude academic outcomes, demographic codes and other unused fields before fitting K-Means.</div>
</div>

In [ ]:
CLUSTER_FEATURES = [
    'StudyHours', 'Attendance', 'Resources', 'Motivation',
    'OnlineCourses', 'Discussions', 'AssignmentCompletion',
    'EduTech', 'StressLevel',
]
EXCLUDED_FROM_CLUSTERING = [
    col for col in EXPECTED_COLUMNS if col not in CLUSTER_FEATURES
]

X = working_df[CLUSTER_FEATURES].copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('Nine modelling variables:')
print(', '.join(CLUSTER_FEATURES))
print('Excluded from the clustering model:')
print(', '.join(EXCLUDED_FROM_CLUSTERING))
print('Scaled feature matrix:', X_scaled.shape)

<div style="background-color:#F3F7FA;border-left:4px solid #476B82;border-radius:5px;padding:11px 14px;margin:15px 0 7px;">
<strong style="color:#253F50;">Figure 4.6 | Elbow analysis</strong><div style="font-size:0.94em;color:#485A65;margin-top:3px;">Compare within-cluster sum of squares across candidate cluster counts to assess diminishing improvement.</div>
</div>

In [ ]:
wcss = []
cluster_models = {}

for k in K_CANDIDATES:
    candidate = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    candidate.fit(X_scaled)
    cluster_models[k] = candidate
    wcss.append(candidate.inertia_)

fig, ax = plt.subplots(figsize=(8.2, 4.8))
ax.plot(K_CANDIDATES, wcss, 'o-', linewidth=1.8, color='#315F79')
ax.axvline(K_SELECTED, linestyle='--', color='#907D61', linewidth=1.2,
           label='Chapter selection: k = 6')
ax.set(title='Elbow Method for Choosing the Number of Clusters',
       xlabel='Number of Clusters (k)', ylabel='Inertia (WCSS)', xticks=K_CANDIDATES)
ax.legend(frameon=False)
fig.tight_layout()
save_figure(fig, '4_6', 'elbow_method')

<div style="background-color:#F3F7FA;border-left:4px solid #476B82;border-radius:5px;padding:11px 14px;margin:15px 0 7px;">
<strong style="color:#253F50;">Figure 4.7 | Silhouette analysis</strong><div style="font-size:0.94em;color:#485A65;margin-top:3px;">Evaluate cluster separation using a reproducible sample to avoid the quadratic cost of full pairwise comparisons.</div>
</div>

In [ ]:
silhouette_values = []
n_observations = len(working_df)
sample_size = min(SILHOUETTE_SAMPLE_SIZE, n_observations)

for k in K_CANDIDATES:
    labels = cluster_models[k].labels_
    value = silhouette_score(
        X_scaled, labels, metric='euclidean', sample_size=sample_size,
        random_state=RANDOM_STATE,
    )
    silhouette_values.append(value)

fig, ax = plt.subplots(figsize=(8.2, 4.8))
ax.plot(K_CANDIDATES, silhouette_values, 's-', linewidth=1.8, color='#46796E')
ax.axvline(K_SELECTED, linestyle='--', color='#907D61', linewidth=1.2,
           label='Chapter selection: k = 6')
ax.set(title='Silhouette Scores Across k', xlabel='Number of Clusters (k)',
       ylabel='Average Silhouette Score', xticks=K_CANDIDATES)
ax.legend(frameon=False)
fig.tight_layout()
save_figure(fig, '4_7', 'silhouette_scores')

print(f'Silhouette evaluation: random sample of {sample_size:,} records out of {n_observations:,}.')
print('This sampling choice may yield small differences from a full-data calculation.')
display(pd.DataFrame({'k': K_CANDIDATES, 'WCSS': wcss,
                      'Silhouette (sampled)': silhouette_values}).round(4))

## 3. Six-cluster model and visual interpretation

<div style="background-color:#F3F7FA;border-left:4px solid #476B82;border-radius:5px;padding:11px 14px;margin:15px 0 7px;">
<strong style="color:#253F50;">Final K-Means model (k = 6)</strong><div style="font-size:0.94em;color:#485A65;margin-top:3px;">Use the six-cluster solution selected in the book and calculate each cluster’s standardised behavioural profile.</div>
</div>

In [ ]:
final_model = cluster_models[K_SELECTED]
cluster_assignments = final_model.labels_
profile_df = working_df.copy()
profile_df['Cluster'] = cluster_assignments
centroid_df = pd.DataFrame(
    final_model.cluster_centers_, columns=CLUSTER_FEATURES,
    index=pd.Index(range(K_SELECTED), name='Cluster')
)

print('Cluster sizes:')
display(profile_df['Cluster'].value_counts().sort_index().to_frame('Records'))
print('Standardised feature centroids:')
display(centroid_df.round(2))

<div style="background-color:#F3F7FA;border-left:4px solid #476B82;border-radius:5px;padding:11px 14px;margin:15px 0 7px;">
<strong style="color:#253F50;">Figure 4.8 | PCA projection</strong><div style="font-size:0.94em;color:#485A65;margin-top:3px;">Project the nine-dimensional standardised data into two dimensions for visual inspection, not for refitting the clusters.</div>
</div>

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
coordinates = pca.fit_transform(X_scaled)
plot_df = pd.DataFrame({
    'PC1': coordinates[:, 0], 'PC2': coordinates[:, 1],
    'Cluster': cluster_assignments.astype(str),
})

fig, ax = plt.subplots(figsize=(9, 6))
sns.scatterplot(data=plot_df, x='PC1', y='PC2', hue='Cluster',
                palette='viridis', alpha=0.37, s=13, linewidth=0, ax=ax)
ax.set(title='PCA Projection of Student Clusters',
       xlabel=f'Principal Component 1 ({100*pca.explained_variance_ratio_[0]:.1f}%)',
       ylabel=f'Principal Component 2 ({100*pca.explained_variance_ratio_[1]:.1f}%)')
ax.legend(title='Cluster', bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False)
fig.tight_layout()
save_figure(fig, '4_8', 'pca_cluster_projection')

<div style="background-color:#F3F7FA;border-left:4px solid #476B82;border-radius:5px;padding:11px 14px;margin:15px 0 7px;">
<strong style="color:#253F50;">Figure 4.9 | Cluster profile heatmap</strong><div style="font-size:0.94em;color:#485A65;margin-top:3px;">Compare cluster centroids on a common standardised scale to identify relative behavioural and contextual patterns.</div>
</div>

In [ ]:
fig, ax = plt.subplots(figsize=(10.5, 6))
sns.heatmap(centroid_df.T, cmap='vlag', center=0, annot=True,
            fmt='.2f', linewidths=0.4, cbar_kws={'label': 'Standardised centroid'}, ax=ax)
ax.set(title='Cluster Profiles (Standardised Feature Patterns)',
       xlabel='Cluster ID', ylabel='Behavioural Features')
fig.tight_layout()
save_figure(fig, '4_9', 'standardised_cluster_profiles')

## 4. Academic outcomes examined after clustering

<div style="background-color:#F3F7FA;border-left:4px solid #476B82;border-radius:5px;padding:11px 14px;margin:15px 0 7px;">
<strong style="color:#253F50;">Figure 4.10 | Post-hoc exam scores</strong><div style="font-size:0.94em;color:#485A65;margin-top:3px;">Compare exam scores after clustering. Academic outcomes do not contribute to the cluster assignments.</div>
</div>

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5))
sns.boxplot(data=profile_df, x='Cluster', y='ExamScore', color='#9FC4CC', ax=ax)
ax.set(title='Exam Scores by Cluster (Post-hoc Profiling)',
       xlabel='Learner Cluster', ylabel='Exam Score')
fig.tight_layout()
save_figure(fig, '4_10', 'exam_score_by_cluster')

<div style="background-color:#F3F7FA;border-left:4px solid #476B82;border-radius:5px;padding:11px 14px;margin:15px 0 7px;">
<strong style="color:#253F50;">Figure 4.11 | Post-hoc grade distributions</strong><div style="font-size:0.94em;color:#485A65;margin-top:3px;">Display the distribution of grade codes by cluster without using grades to define the clusters.</div>
</div>

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 5))
sns.countplot(data=profile_df, x='Cluster', hue='FinalGrade',
              palette='mako', ax=ax)
ax.set(title='Final Grade Distribution by Cluster',
       xlabel='Learner Cluster', ylabel='Number of Records')
ax.legend(title='Final Grade (code)', bbox_to_anchor=(1.02, 1),
          loc='upper left', frameon=False)
fig.tight_layout()
save_figure(fig, '4_11', 'final_grade_by_cluster')

<div style="background-color:#F3F7FA;border-left:4px solid #476B82;border-radius:5px;padding:11px 14px;margin:15px 0 7px;">
<strong style="color:#253F50;">Figure 4.12 | Six-cluster radar profiles</strong><div style="font-size:0.94em;color:#485A65;margin-top:3px;">Display nine standardised characteristics for each cluster in a single six-panel figure.</div>
</div>

In [ ]:
from math import pi

SHORT_LABELS = [
    'Study', 'Attend.', 'Resources', 'Motivation', 'Online',
    'Discussion', 'Completion', 'EduTech', 'Stress',
]
angles = np.linspace(0, 2*pi, len(CLUSTER_FEATURES), endpoint=False)
closed_angles = np.r_[angles, angles[0]]
cluster_colors = sns.color_palette('viridis', K_SELECTED)

fig, axes = plt.subplots(2, 3, figsize=(17, 10.5),
                         subplot_kw={'projection': 'polar'})
for cluster_id, ax in enumerate(axes.ravel()):
    values = centroid_df.loc[cluster_id].to_numpy()
    closed_values = np.r_[values, values[0]]
    ax.set_theta_offset(pi/2)
    ax.set_theta_direction(-1)
    ax.plot(closed_angles, closed_values, color=cluster_colors[cluster_id], linewidth=1.7)
    ax.fill(closed_angles, closed_values, color=cluster_colors[cluster_id], alpha=0.16)
    ax.set_xticks(angles)
    ax.set_xticklabels(SHORT_LABELS, fontsize=9)
    ax.set_rlabel_position(18)
    ax.tick_params(axis='y', labelsize=8)
    ax.set_ylim(-2.5, 2.5)
    ax.set_title(f'Cluster {cluster_id}', y=1.17, fontweight='bold')

fig.suptitle('Standardised Multidimensional Learner Profiles', fontsize=15, y=1.02)
fig.subplots_adjust(wspace=0.38, hspace=0.60)
save_figure(fig, '4_12', 'six_cluster_radar_profiles')

## 5. Descriptive interpretation and reproducibility

<div style="background-color:#F3F7FA;border-left:4px solid #476B82;border-radius:5px;padding:11px 14px;margin:15px 0 7px;">
<strong style="color:#253F50;">Descriptive cluster summary</strong><div style="font-size:0.94em;color:#485A65;margin-top:3px;">Review selected behavioural characteristics and post-hoc outcomes without treating cluster membership as a permanent student label.</div>
</div>

In [ ]:
summary = (
    profile_df.groupby('Cluster')
    .agg(Records=('StudyHours', 'size'),
         MeanStudyHours=('StudyHours', 'mean'),
         MeanAttendance=('Attendance', 'mean'),
         MeanAssignmentCompletion=('AssignmentCompletion', 'mean'),
         MeanStressLevelCode=('StressLevel', 'mean'),
         MeanExamScore=('ExamScore', 'mean'))
    .round(2)
)
print('Descriptive cluster summary:')
display(summary)
print('Educational interpretation requires examination of individual context,')
print('measurement limitations, and professional judgement.')

<div style="background-color:#F3F7FA;border-left:4px solid #476B82;border-radius:5px;padding:11px 14px;margin:15px 0 7px;">
<strong style="color:#253F50;">Export reproducible outputs</strong><div style="font-size:0.94em;color:#485A65;margin-top:3px;">Save the observational data with cluster identifiers and verify that the eleven numbered figures were generated.</div>
</div>

In [ ]:
export_file = OUTPUT_DIR / 'chapter04_cluster_assignments.csv'
export_df = student_df.copy()
export_df['Cluster'] = cluster_assignments
export_df.to_csv(export_file, index=False)

print('Cluster assignments:', export_file)
print('Exported figures:', len(list(OUTPUT_DIR.glob('figure_*.pdf'))), 'PDF files')
print('Expected book figures: 4.2–4.12 (11 plots)')

### Reading the results

The **Elbow** and **Silhouette** plots help assess candidate partitions, but neither alone proves that six discrete learner types exist. The **PCA projection** is a two-dimensional visual summary; cluster overlap should be interpreted cautiously. The **profile heatmap** and **radar plots** describe relative feature patterns, while **exam scores and grades** are examined only after grouping. Educational support strategies should be based on contextual interpretation rather than automatic categorisation.

**Reproducibility note:** K-Means uses `random_state=42`, `n_init=10`, and nine standardised features, with `k=6` as specified in the chapter. Silhouette scores are calculated using a fixed random sample of up to 2,500 observations to keep execution practical; these values need not exactly match results calculated using the full dataset. Different software versions or different downloaded dataset versions may also change individual scores and plotted values.